# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [2]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


GET columns names from each table so that we only load the required ones for Ranking signal analysis

In [6]:
print('Column names for fact_daily:')
print(con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 0").columns)

print('\nColumn names for fact_query_90d:')
print(con.sql(f"SELECT * FROM {TABLES['fact_query_90d']} LIMIT 0").columns)

Column names for fact_daily:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Column names for fact_query_90d:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressio

In [7]:
print('\nColumn names for dim_clients:')
print(con.sql(f"SELECT * FROM {TABLES['dim_clients']} LIMIT 0").columns)

print('\nColumn names for dim_content:')
print(con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 0").columns)


Column names for dim_clients:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

Column names for dim_content:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224


# **Get Client data information**
First find which clients have atleast 90 day data and then filter them out so that we can split the data in same time window

In [5]:
overall_end_date = con.sql(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
print(f"The overall end date (MAX report_date from fact_daily) is: {overall_end_date}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

The overall end date (MAX report_date from fact_daily) is: 2026-06-30


In [8]:
client_end_dates = con.sql(f"""
    SELECT client_hash_id, MAX(report_date) AS client_end_date
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
""").df()

clients_with_end_date = clients.merge(client_end_dates, on='client_hash_id', how='left')
print("Clients with their respective end dates (MAX report_date from fact_daily):")
display(clients_with_end_date.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with their respective end dates (MAX report_date from fact_daily):


,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30


In [9]:
clients_with_end_date['data_history_duration'] = clients_with_end_date['client_end_date'] - clients_with_end_date['gsc_data_start']
display(clients_with_end_date.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30,504 days
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30,476 days
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30,388 days


In [10]:
import pandas as pd

clients_less_than_90_days = clients_with_end_date[clients_with_end_date['data_history_duration'] < pd.Timedelta(days=90)]
display(clients_less_than_90_days.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
57,client_8ddc46da5414ffd8,gsc_only,2026-04-07,NaT,2026-06-30,84 days
58,client_06d356715a8ff3b6,gsc_and_ga4,2026-04-10,2026-04-06,2026-06-30,81 days
59,client_0b245132bb722950,gsc_and_ga4,2026-04-12,2026-04-24,2026-06-30,79 days
60,client_9c26c096d6e57253,gsc_and_ga4,2026-04-29,2026-04-23,2026-06-30,62 days
61,client_c353557474475e51,gsc_and_ga4,2026-04-30,2026-06-01,2026-06-30,61 days


In [11]:
clients_with_missing_values = clients_with_end_date[clients_with_end_date.isnull().any(axis=1)]
display(clients_with_missing_values.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30,388 days
14,client_8ae2bfb5aa1ffa1e,gsc_only,2025-07-28,NaT,2026-06-30,337 days
15,client_8dbf3abdf07569e0,gsc_only,2025-07-29,NaT,2026-06-30,336 days
16,client_08a6a72ff48e62c0,gsc_only,2025-09-24,NaT,2026-06-30,279 days
22,client_795153d5b7850ccf,gsc_only,2025-09-24,NaT,2026-06-30,279 days


In [12]:
clients_to_exclude = pd.concat([
    clients_with_missing_values['client_hash_id'],
    clients_less_than_90_days['client_hash_id']
]).unique()

cleaned_clients = clients_with_end_date[
    ~clients_with_end_date['client_hash_id'].isin(clients_to_exclude)
]

print(f"Number of clients after cleaning: {len(cleaned_clients)}")
display(cleaned_clients.head())

Number of clients after cleaning: 41


,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30,504 days
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30,476 days
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15,2026-06-30,377 days


Then Select 2 clients from the cleand_clients and get their data from dim_content. I only select those features which are required for ranking signal analysis

In [13]:
# Select the first two client_hash_id from cleaned_clients
two_client_ids = cleaned_clients['client_hash_id'].iloc[0:2].tolist()

print(f"Selected clients for analysis: {two_client_ids}")

# Prepare the client IDs for the SQL IN clause
quoted_two_client_ids = [f"'{_id}'" for _id in two_client_ids]

# Define the columns to extract from dim_content
content_cols = [
    'client_hash_id',
    'content_hash_id',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'char_count', # Corrected from 'character_count'
    'main_intent',
    'keyword_char_count',
    'keyword_token_count',
    'url_char_count',
    'backlinks',
    'category_count',
    'word_count',
    'content_type',
    'provider_used',
    'model_used',
    'content_created_date',
    'content_updated_date'
]

# Construct the SQL query to get data for the two selected clients and specific columns
sql_query = f"""
    SELECT {', '.join(content_cols)}
    FROM {TABLES['dim_content']}
    WHERE client_hash_id IN ({', '.join(quoted_two_client_ids)});
"""

# Execute the query and load into a pandas DataFrame
data_for_two_clients = con.sql(sql_query).df()

print(f"Number of content items for selected clients: {len(data_for_two_clients):,}")
display(data_for_two_clients.head())

Selected clients for analysis: ['client_9958f0a7ae1df715', 'client_ff644d8251367cbb']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of content items for selected clients: 7,598


,client_hash_id,content_hash_id,search_volume,competition,competition_level,cpc,char_count,main_intent,keyword_char_count,keyword_token_count,url_char_count,backlinks,category_count,word_count,content_type,provider_used,model_used,content_created_date,content_updated_date
0,client_9958f0a7ae1df715,content_000005d4ced12088,110,0.03,LOW,0.06,<NA>,commercial,43,8,79,<NA>,0,<NA>,keyword article,None,gpt-4o-mini,2025-03-28,2026-05-18
1,client_9958f0a7ae1df715,content_0002bd310bf01f15,<NA>,NaN,None,NaN,<NA>,informational,29,6,106,<NA>,0,<NA>,keyword article,None,gpt-4o-mini,2025-01-08,2026-05-18
2,client_9958f0a7ae1df715,content_000d3f2ab6f6e376,10,0.00,LOW,0.00,<NA>,transactional,29,5,87,<NA>,0,<NA>,keyword article,None,gpt-4o-mini,2025-03-28,2026-05-18
3,client_9958f0a7ae1df715,content_001a69e9d74a62bf,<NA>,NaN,None,NaN,<NA>,commercial,26,5,116,<NA>,0,<NA>,keyword article,None,gpt-4o-mini,2025-01-08,2026-05-18
4,client_9958f0a7ae1df715,content_002c2eb47c74cb07,0,0.00,LOW,0.00,<NA>,informational,39,7,75,<NA>,0,<NA>,keyword article,None,gpt-4o-mini,2025-03-28,2026-05-18


In [14]:
# Ensure date columns are in datetime format
data_for_two_clients['content_created_date'] = pd.to_datetime(data_for_two_clients['content_created_date'])
data_for_two_clients['content_updated_date'] = pd.to_datetime(data_for_two_clients['content_updated_date'])

# Convert overall_end_date to datetime for calculations
overall_end_date_dt = pd.to_datetime(overall_end_date)

# Calculate content_age_days
data_for_two_clients['content_age_days'] = (overall_end_date_dt - data_for_two_clients['content_created_date']).dt.days

# Calculate days_since_last_update
data_for_two_clients['days_since_last_update'] = (overall_end_date_dt - data_for_two_clients['content_updated_date']).dt.days

print("Calculated 'content_age_days' and 'days_since_last_update'.")
display(data_for_two_clients[['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'content_age_days', 'days_since_last_update']].head())

Calculated 'content_age_days' and 'days_since_last_update'.


,client_hash_id,content_hash_id,content_created_date,content_updated_date,content_age_days,days_since_last_update
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-28,2026-05-18,459,43
1,client_9958f0a7ae1df715,content_0002bd310bf01f15,2025-01-08,2026-05-18,538,43
2,client_9958f0a7ae1df715,content_000d3f2ab6f6e376,2025-03-28,2026-05-18,459,43
3,client_9958f0a7ae1df715,content_001a69e9d74a62bf,2025-01-08,2026-05-18,538,43
4,client_9958f0a7ae1df715,content_002c2eb47c74cb07,2025-03-28,2026-05-18,459,43


In [ ]:
# The original date columns have already been removed in a previous step when creating `data_for_two_clients`.
# The new derived features are now present and can be used for further analysis.

In [15]:
print("Missing values per feature for each of the two selected clients (least missing data):")
for client_id in data_for_two_clients['client_hash_id'].unique():
    client_data = data_for_two_clients[data_for_two_clients['client_hash_id'] == client_id]
    missing_values_percentage = client_data.isnull().sum() * 100 / len(client_data)
    missing_values_percentage = missing_values_percentage[missing_values_percentage > 0].sort_values(ascending=False)

    if not missing_values_percentage.empty:
        print(f"\nClient: {client_id}")
        display(missing_values_percentage.to_frame(name='Missing Percentage'))
    else:
        print(f"\nClient: {client_id} has no missing values.")

Missing values per feature for each of the two selected clients (least missing data):

Client: client_9958f0a7ae1df715


,Missing Percentage
provider_used,100.000000
word_count,68.465540
char_count,68.465540
backlinks,64.076723
search_volume,14.369311
cpc,14.369311
competition_level,14.369311
competition,14.369311
main_intent,0.065020



Client: client_ff644d8251367cbb


,Missing Percentage
provider_used,100.000000
word_count,92.790801
char_count,92.790801
backlinks,84.542238
search_volume,6.081380
cpc,6.081380
competition_level,6.081380
competition,6.081380
main_intent,0.044228


### Selecting the two clients with the least missing data

In the above section for the 2 clients the percentage of missing data is very high so I select the 2 clients with least missing data

In [16]:
# Get all client_hash_ids from cleaned_clients
all_cleaned_client_ids = cleaned_clients['client_hash_id'].tolist()
quoted_all_cleaned_client_ids = [f"'{_id}'" for _id in all_cleaned_client_ids]

# Define the columns to extract from dim_content (same as before to maintain consistency)
content_cols = [
    'client_hash_id',
    'content_hash_id',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'char_count',
    'main_intent',
    'keyword_char_count',
    'keyword_token_count',
    'url_char_count',
    'backlinks',
    'category_count',
    'word_count',
    'content_type',
    'provider_used',
    'model_used',
    'content_created_date',
    'content_updated_date'
]

# Construct the SQL query to get data for all cleaned clients and specific columns
sql_query_all_cleaned = f"""
    SELECT {', '.join(content_cols)}
    FROM {TABLES['dim_content']}
    WHERE client_hash_id IN ({', '.join(quoted_all_cleaned_client_ids)});
"""

# Execute the query and load into a pandas DataFrame
# This might be a large query but avoids multiple small queries
all_cleaned_clients_content_data = con.sql(sql_query_all_cleaned).df()

print(f"Number of content items for all cleaned clients: {len(all_cleaned_clients_content_data):,}")
display(all_cleaned_clients_content_data.head())

Number of content items for all cleaned clients: 301,638


,client_hash_id,content_hash_id,search_volume,competition,competition_level,cpc,char_count,main_intent,keyword_char_count,keyword_token_count,url_char_count,backlinks,category_count,word_count,content_type,provider_used,model_used,content_created_date,content_updated_date
0,client_3197e6291363b4db,content_4afb6016ac96c960,0,0.00,LOW,0.00,9347,informational,32,7,123,0,0,1490,keyword article,None,gpt-4o-mini,2025-12-18,2026-06-01
1,client_3197e6291363b4db,content_4b06b437b055bd2c,1000,0.05,LOW,0.06,22903,commercial,30,6,101,<NA>,0,3899,keyword article,None,gpt-4o-mini,2025-06-26,2026-05-20
2,client_3197e6291363b4db,content_4b2570a44c727567,0,0.00,LOW,0.00,8294,informational,48,10,97,0,0,1337,keyword article,None,gpt-4o-mini,2025-12-17,2026-06-01
3,client_3197e6291363b4db,content_4b278b8efdba8829,60500,0.36,MEDIUM,0.23,21883,commercial,28,5,126,<NA>,0,3638,keyword article,None,gpt-4o-mini,2025-06-26,2026-05-20
4,client_3197e6291363b4db,content_4b2c2948bd4dcd49,10,0.00,LOW,0.00,13657,informational,35,7,86,60,3,2139,keyword article,None,gpt-4o-mini,2025-09-07,2026-05-20


In [17]:
# Calculate the overall missing percentage for each client
missing_data_summary = []
for client_id in all_cleaned_clients_content_data['client_hash_id'].unique():
    client_data_subset = all_cleaned_clients_content_data[all_cleaned_clients_content_data['client_hash_id'] == client_id]
    total_missing_cells = client_data_subset.isnull().sum().sum()
    total_cells = client_data_subset.shape[0] * client_data_subset.shape[1]
    if total_cells > 0:
        overall_missing_percentage = (total_missing_cells / total_cells) * 100
    else:
        overall_missing_percentage = 0
    missing_data_summary.append({'client_hash_id': client_id, 'overall_missing_percentage': overall_missing_percentage})

missing_data_df = pd.DataFrame(missing_data_summary).sort_values(by='overall_missing_percentage', ascending=True)
print("Clients sorted by overall missing data percentage:")
display(missing_data_df.head())

# Select the two clients with the least missing data
best_two_client_ids = missing_data_df.head(2)['client_hash_id'].tolist()
print(f"\nThe two clients with the least missing data are: {best_two_client_ids}")

Clients sorted by overall missing data percentage:


,client_hash_id,overall_missing_percentage
24,client_ccdd78843409c8c7,0.000000
36,client_7eafe750768f0ac2,0.000000
40,client_f6f0cdf26d03d7bd,0.000000
18,client_86ebc2f12c01f586,0.017593
32,client_b77d0d5f08f05e64,0.017715



The two clients with the least missing data are: ['client_ccdd78843409c8c7', 'client_7eafe750768f0ac2']


In [18]:
# Update data_for_two_clients with data for the newly selected best two clients
data_for_two_clients = all_cleaned_clients_content_data[
    all_cleaned_clients_content_data['client_hash_id'].isin(best_two_client_ids)
]

print(f"Number of content items for the newly selected two clients: {len(data_for_two_clients):,}")
display(data_for_two_clients.head())

# Re-calculate derived date features for these new clients (copied from 100cf134)
# Ensure date columns are in datetime format
data_for_two_clients['content_created_date'] = pd.to_datetime(data_for_two_clients['content_created_date'])
data_for_two_clients['content_updated_date'] = pd.to_datetime(data_for_two_clients['content_updated_date'])

# Convert overall_end_date to datetime for calculations (overall_end_date is already defined in the kernel)
overall_end_date_dt = pd.to_datetime(overall_end_date)

# Calculate content_age_days
data_for_two_clients['content_age_days'] = (overall_end_date_dt - data_for_two_clients['content_created_date']).dt.days

# Calculate days_since_last_update
data_for_two_clients['days_since_last_update'] = (overall_end_date_dt - data_for_two_clients['content_updated_date']).dt.days

print("Recalculated 'content_age_days' and 'days_since_last_update' for the new two clients.")
display(data_for_two_clients[['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'content_age_days', 'days_since_last_update']].head())

Number of content items for the newly selected two clients: 1,865


,client_hash_id,content_hash_id,search_volume,competition,competition_level,cpc,char_count,main_intent,keyword_char_count,keyword_token_count,url_char_count,backlinks,category_count,word_count,content_type,provider_used,model_used,content_created_date,content_updated_date
121636,client_ccdd78843409c8c7,content_011fbefa7aa02126,40,0.01,LOW,0.00,19968,informational,45,7,97,325,2,2834,keyword article,google,gemini-3-flash-preview,2026-02-16,2026-05-20
121637,client_ccdd78843409c8c7,content_01d35f449f93cf8d,10,0.06,LOW,0.00,19004,commercial,34,6,103,31,8,2674,keyword article,google,gemini-3-flash-preview,2026-02-24,2026-07-01
121638,client_ccdd78843409c8c7,content_0224559ef129d4cd,10,0.29,LOW,0.00,19426,commercial,37,6,105,0,5,2799,keyword article,google,gemini-3-flash-preview,2026-02-24,2026-07-01
121639,client_ccdd78843409c8c7,content_024fb066015e0218,10,0.61,MEDIUM,11.01,18795,informational,35,6,97,0,10,2672,keyword article,google,gemini-3-flash-preview,2026-02-24,2026-07-01
121640,client_ccdd78843409c8c7,content_028ea0e57b7919e0,10,0.17,LOW,0.00,20302,navigational,42,6,106,436,5,2807,keyword article,google,gemini-3-flash-preview,2026-02-24,2026-07-01


Recalculated 'content_age_days' and 'days_since_last_update' for the new two clients.


/tmp/ipykernel_2153/2404559138.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_for_two_clients['content_created_date'] = pd.to_datetime(data_for_two_clients['content_created_date'])
/tmp/ipykernel_2153/2404559138.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_for_two_clients['content_updated_date'] = pd.to_datetime(data_for_two_clients['content_updated_date'])
/tmp/ipykernel_2153/2404559138.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

,client_hash_id,content_hash_id,content_created_date,content_updated_date,content_age_days,days_since_last_update
121636,client_ccdd78843409c8c7,content_011fbefa7aa02126,2026-02-16,2026-05-20,134,41
121637,client_ccdd78843409c8c7,content_01d35f449f93cf8d,2026-02-24,2026-07-01,126,-1
121638,client_ccdd78843409c8c7,content_0224559ef129d4cd,2026-02-24,2026-07-01,126,-1
121639,client_ccdd78843409c8c7,content_024fb066015e0218,2026-02-24,2026-07-01,126,-1
121640,client_ccdd78843409c8c7,content_028ea0e57b7919e0,2026-02-24,2026-07-01,126,-1


In [19]:
# Save the data_for_two_clients DataFrame to a CSV file
data_for_two_clients.to_csv('/content/data_for_two_clients.csv', index=False)

print("DataFrame 'data_for_two_clients' saved to '/content/data_for_two_clients.csv'")

DataFrame 'data_for_two_clients' saved to '/content/data_for_two_clients.csv'


In [20]:
# Prepare the best two client IDs for the SQL IN clause
quoted_best_two_client_ids = [f"'{_id}'" for _id in best_two_client_ids]

# Construct the SQL query to get fact_daily data for the selected two clients
sql_query_fact_daily = f"""
    SELECT client_hash_id, content_hash_id, report_date, gsc_avg_position
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id IN ({', '.join(quoted_best_two_client_ids)});
"""

# Execute the query and load into a pandas DataFrame
fact_daily_for_two_clients = con.sql(sql_query_fact_daily).df()

print(f"Number of daily fact records for the selected two clients: {len(fact_daily_for_two_clients):,}")
display(fact_daily_for_two_clients.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of daily fact records for the selected two clients: 171,653


,client_hash_id,content_hash_id,report_date,gsc_avg_position
0,client_ccdd78843409c8c7,content_ffef8999f9794511,2026-02-12,NaN
1,client_ccdd78843409c8c7,content_6a779b34473713ea,2026-02-12,NaN
2,client_ccdd78843409c8c7,content_a384f81f20389779,2026-02-12,NaN
3,client_ccdd78843409c8c7,content_3dbaaac8e347290b,2026-02-12,NaN
4,client_ccdd78843409c8c7,content_2be76e72591d0730,2026-02-12,NaN


In [21]:
# Merge data_for_two_clients with fact_daily_for_two_clients to get gsc_avg_position
data_for_two_clients = pd.merge(
    data_for_two_clients,
    fact_daily_for_two_clients[['client_hash_id', 'content_hash_id', 'gsc_avg_position']],
    on=['client_hash_id', 'content_hash_id'],
    how='left'
).drop_duplicates(subset=['client_hash_id', 'content_hash_id']) # Drop duplicates after merge if any

# Filter out rows where 'gsc_avg_position' is NaN
original_row_count = len(data_for_two_clients)
data_for_two_clients_cleaned = data_for_two_clients.dropna(subset=['gsc_avg_position'])

# Update data_for_two_clients with the cleaned DataFrame
data_for_two_clients = data_for_two_clients_cleaned

# Print information about the removal
removed_rows = original_row_count - len(data_for_two_clients)
print(f"Removed {removed_rows} rows due to missing 'gsc_avg_position'.")
print(f"Remaining rows in data_for_two_clients: {len(data_for_two_clients)}")

display(data_for_two_clients.head())

Removed 1851 rows due to missing 'gsc_avg_position'.
Remaining rows in data_for_two_clients: 14


,client_hash_id,content_hash_id,search_volume,competition,competition_level,cpc,char_count,main_intent,keyword_char_count,keyword_token_count,...,category_count,word_count,content_type,provider_used,model_used,content_created_date,content_updated_date,content_age_days,days_since_last_update,gsc_avg_position
55903,client_7eafe750768f0ac2,content_01d903cee2665ce2,0,0.00,LOW,0.0,19751,informational,29,4,...,0,2807,keyword article,google,gemini-3-flash-preview,2026-03-12,2026-05-20,110,41,17.0
59398,client_7eafe750768f0ac2,content_094259d13e865b9e,0,0.00,LOW,0.0,21336,informational,39,10,...,0,3116,keyword article,google,gemini-3-flash-preview,2026-02-25,2026-07-01,125,-1,3.0
64273,client_7eafe750768f0ac2,content_14e4334e8333f742,10,0.00,LOW,0.0,18641,informational,35,7,...,4,2597,keyword article,google,gemini-3-flash-preview,2026-02-25,2026-07-01,125,-1,11.5
70796,client_7eafe750768f0ac2,content_21654c4de6e9cf5b,10,0.64,MEDIUM,0.0,17996,commercial,30,3,...,3,2559,keyword article,google,gemini-3-flash-preview,2026-04-13,2026-05-20,78,41,7.0
73349,client_7eafe750768f0ac2,content_262827d4adea47f6,0,0.00,LOW,0.0,19538,informational,48,8,...,0,2830,keyword article,google,gemini-3-flash-preview,2026-02-25,2026-07-01,125,-1,8.0


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
